In [86]:
import pandas as pd
import numpy as np 
from scipy.spatial.distance import cdist

In [158]:
historical_perf = pd.read_parquet('../data/processed/historical_performance_long.parquet')

In [168]:
plan_df = pd.read_csv('./../test_plans.csv')

In [169]:
df = pd.read_parquet('../data/views/vw_ATD_dashboard.parquet')

In [164]:
experiences = ['novice', 'low', 'medium', 'high', 'expert', 'unassigned']
courier_flows = ['Fleet', 'Logistics', 'Motorbike', 'SUV', 'UberX', 'UberEats', 'Onboarder']

results = []

In [165]:
historical_perf = historical_perf.set_index(['territory', 'day_of_week', 'hour'], drop=False)

In [166]:
historical_perf

territory  day_of_week  hour   %novice      %low  \
territory day_of_week hour                                                    
Central   0           0      Central            0     0  0.076923  0.145749   
                      1      Central            0     1  0.056604  0.141509   
                      2      Central            0     2  0.047619  0.063492   
                      3      Central            0     3  0.120000  0.080000   
                      4      Central            0     4  0.062500  0.125000   
...                              ...          ...   ...       ...       ...   
West      6           19        West            6    19  0.098191  0.131045   
                      20        West            6    20  0.105112  0.122290   
                      21        West            6    21  0.096841  0.135302   
                      22        West            6    22  0.073209  0.137072   
                      23        West            6    23  0.114815  0.092593   

                             %medium     %high   %expert  %unassigned  \
territory day_of_week hour                                              
Central   0           0     0.259109  0.327935  0.190283          0.0   
                      1     0.264151  0.415094  0.122642          0.0   
                      2     0.444444  0.206349  0.238095          0.0   
                      3     0.160000  0.360000  0.280000          0.0   
                      4     0.062500  0.437500  0.312500          0.0   
...                              ...       ...       ...          ...   
West      6           19    0.231820  0.333702  0.205242          0.0   
                      20    0.231084  0.335378  0.206135          0.0   
                      21    0.223214  0.358516  0.186126          0.0   
                      22    0.249221  0.323988  0.216511          0.0   
                      23    0.188889  0.448148  0.155556          0.0   

                              %Fleet  ...  %Onboarder      %SUV  %UberEats  \
territory day_of_week hour            ...                                    
Central   0           0     0.012146  ...         0.0  0.000000   0.064777   
                      1     0.009434  ...         0.0  0.000000   0.122642   
                      2     0.000000  ...         0.0  0.000000   0.063492   
                      3     0.040000  ...         0.0  0.000000   0.120000   
                      4     0.000000  ...         0.0  0.000000   0.187500   
...                              ...  ...         ...       ...        ...   
West      6           19    0.001846  ...         0.0  0.000000   0.036545   
                      20    0.001227  ...         0.0  0.002045   0.033947   
                      21    0.000000  ...         0.0  0.002060   0.037775   
                      22    0.001558  ...         0.0  0.001558   0.052960   
                      23    0.003704  ...         0.0  0.000000   0.059259   

                              %UberX  total_orders  drivers_per_day  \
territory day_of_week hour                                            
Central   0           0     0.000000           247            91213   
                      1     0.000000           106            91213   
                      2     0.031746            63            91213   
                      3     0.040000            25            91213   
                      4     0.000000            16            91213   
...                              ...           ...              ...   
West      6           19    0.001846          2709            91213   
                      20    0.002045          2445            91213   
                      21    0.003434          1456            91213   
                      22    0.000000           642            91213   
                      23    0.011111           270            91213   

                            median_ATD     p95_ATD  pct_breaches   breach_cost  
territory day_of_week hour   

In [170]:
for _, row in plan_df.iterrows():
    territory = row["territory"]
    day = row["day_of_week"]
    hour = row["hour"]
    n_drivers = row["drivers_per_day"]

    # Vectorizacion del plan actual
    driver_mix_vector = np.array([row.get(f"%{exp}", 0.0) for exp in experiences])
    courier_mix_vector = np.array([row.get(f"%{flow}", 0.0) for flow in courier_flows])
    plan_vector = np.concatenate([[day], [hour], [n_drivers], driver_mix_vector, courier_mix_vector])

    # tomamos los días y horas a +/- 2 y +/- 1 respectivamente de distancia
    # mod 7 porque así seguimos con la cintinuidad entre 0-6 y 0-23
    valid_days = [(day + i) % 7 for i in [-1, 0, 1]]
    valid_hours = [(hour + i) % 24 for i in range(-2, 3)]

    # Índices históricos que cumplen condiciones
    valid_idx = [
        idx for idx in historical_perf.index
        if idx[0] == territory and idx[1] in valid_days and idx[2] in valid_hours
    ] # Identificamos todas las filas del historico que coinciden con el territorio y +/- 1 día, +/- 2 horas

    if not valid_idx:
        continue

    vectors = []
    metric_info = []

    for idx in valid_idx:
        day_i, hour_i = idx[1], idx[2]
        
        hist_row = historical_perf.loc[idx]

        hist_drivers = row['drivers_per_day']

        hist_dm_vector = np.array([hist_row.get(f"%{exp}", 0.0) for exp in experiences])
        hist_courier_vector = np.array([hist_row.get(f"%{flow}", 0.0) for flow in courier_flows])
        hist_plan_vector = np.concatenate([[day_i], [hour_i], [hist_drivers], hist_dm_vector, hist_courier_vector])

        vectors.append(hist_plan_vector)
        
        metrics = historical_perf[
            (historical_perf["territory"] == territory) &
            (historical_perf["day_of_week"] == day_i) &
            (historical_perf["hour"] == hour_i)
        ].iloc[0]

        metric_info.append(metrics)

    vectors = np.vstack(vectors)
    distances = cdist([plan_vector], vectors, metric='euclidean')[0]
    sorted_indices = np.argsort(distances)

    k = min(5, len(sorted_indices))
    weights = 1 / (distances[sorted_indices[:k]] + 1e-5)
    weights /= weights.sum()

    selected_metrics = [metric_info[i] for i in sorted_indices[:k]]
    weighted_metrics = {
        "median_ATD": 0,
        "p95_ATD": 0,
        "pct_breaches": 0,
        "breach_cost": 0,
        "total_orders": 0
    }

    for i, metrics in enumerate(selected_metrics):
        for kpi in weighted_metrics:
            weighted_metrics[kpi] += round(metrics[kpi] * weights[i], 2)

    weighted_metrics["expected_orders"] = weighted_metrics["total_orders"]

    for kpi in weighted_metrics:
        weighted_metrics[kpi] = round(weighted_metrics[kpi], 2)

    results.append({
        "territory": territory,
        "day_of_week": day,
        "hour": hour,
        "drivers_planned": n_drivers,
        **weighted_metrics
    })

In [171]:
results

[{'territory': 'North',
  'day_of_week': 1,
  'hour': 9,
  'drivers_planned': 40,
  'median_ATD': np.float64(33.65),
  'p95_ATD': np.float64(68.8),
  'pct_breaches': np.float64(0.56),
  'breach_cost': np.float64(179146.71),
  'total_orders': np.float64(1276.14),
  'expected_orders': np.float64(1276.14)},
 {'territory': 'West',
  'day_of_week': 3,
  'hour': 12,
  'drivers_planned': 91213,
  'median_ATD': np.float64(38.02),
  'p95_ATD': np.float64(99.5),
  'pct_breaches': np.float64(0.72),
  'breach_cost': np.float64(193945.2),
  'total_orders': np.float64(1157.0),
  'expected_orders': np.float64(1157.0)},
 {'territory': 'Central',
  'day_of_week': 1,
  'hour': 17,
  'drivers_planned': 91213,
  'median_ATD': np.float64(41.24),
  'p95_ATD': np.float64(74.83),
  'pct_breaches': np.float64(0.65),
  'breach_cost': np.float64(554927.03),
  'total_orders': np.float64(2468.02),
  'expected_orders': np.float64(2468.02)},
 {'territory': 'West',
  'day_of_week': 3,
  'hour': 1,
  'drivers_planned'

In [173]:
historical_perf.columns

Index(['territory', 'day_of_week', 'hour', '%novice', '%low', '%medium',
       '%high', '%expert', '%unassigned', '%Fleet', '%Logistics', '%Motorbike',
       '%Onboarder', '%SUV', '%UberEats', '%UberX', 'total_orders',
       'drivers_per_day', 'median_ATD', 'p95_ATD', 'pct_breaches',
       'breach_cost'],
      dtype='object')

In [179]:
historical_perf[((historical_perf.territory == 'North') |
                (historical_perf.territory == 'South East')) &
                (historical_perf.day_of_week.isin([5,6])) &
                (historical_perf.hour.isin([20, 21, 22, 23, 24]))].reset_index(drop = True).sample(10).to_csv('./../test_plans.csv',
                                                                                                              index = False)